CIFAR10を3層のCNNモデルで分類する

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.ticker as ticker
import torchvision

from Optimizers.SVRG import SVRG
from Optimizers.NFG_SVRG import NFG_SVRG
from Optimizers.ASAI_SVRG import ASAI_SVRG

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
LEARNING_RATE = 0.001
BATCH_SIZE = 64
EPOCHS = 5
SEED = 42
KEEP = False

In [ ]:
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True, num_workers = 0)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 0)

class CNN_Model(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = torch.nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = torch.nn.GroupNorm(8, 32)
        self.act1 = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2)

        self.conv2 = torch.nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = torch.nn.GroupNorm(8, 64) 
        self.act2 = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2)

        self.conv3 = torch.nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = torch.nn.GroupNorm(8, 128) 
        self.act3 = torch.nn.ReLU()
        self.pool3 = torch.nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = torch.nn.Linear(128, 100)
        self.act4 = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(100, 10)

    def forward(self, x):
        y = self.act1(self.bn1(self.conv1(x)))
        y = self.pool1(y)

        y = self.act2(self.bn2(self.conv2(y)))
        y = self.pool2(y)

        y = self.act3(self.bn3(self.conv3(y)))
        y = self.pool3(y)

        y = torch.flatten(y, 1)
        y = self.act4(self.fc1(y))
        y = self.fc2(y)
        return y    


print(CNN_Model()(torch.tensor(np.zeros(shape=(1,3,32,32)),dtype=torch.float32)))

import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 10))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title(f"Label: {labels[i].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
ROOT_DIR = "outputs/CIFAR10_CNN3-1_01"
TARGET_DIR = f"{ROOT_DIR}/{LEARNING_RATE}/{BATCH_SIZE}/{EPOCHS}/{SEED}"
os.makedirs(TARGET_DIR, exist_ok=True)

In [ ]:
def epoch(loader, model, criterion, optimizer = None):
    '''
    loader:データローダー
    model:モデル
    criton:目的関数
    optimizer:手法
    '''
    if not optimizer == None:
        model.train()

    else:
        model.eval()

    total_loss, total_correct, total = 0, 0, 0
    pb = tqdm(loader)
    for X, T in pb:
        X, T = X.to(device), T.to(device)

        if not optimizer == None:

            if isinstance(optimizer, SVRG) or isinstance(optimizer, NFG_SVRG) or isinstance(optimizer, ASAI_SVRG):

                optimizer.calc_snapshot_grads(model, X, T, criterion)

            optimizer.zero_grad()
            Y = model(X)
            loss = criterion(Y, T)
            loss.backward()
            optimizer.step()

        else:
            with torch.no_grad():
                Y = model(X)
                loss = criterion(Y, T)

        total_loss += loss.item() * T.size(0)

        pred = Y.argmax(dim=1)
        mini_correct = (pred == T).sum().item() / T.size(0)
        total_correct += (pred == T).sum().item()
        total += T.size(0)

        pb.set_postfix({"loss": loss.item(), "acc": mini_correct})

    epoch_loss = total_loss / total
    epoch_acc = total_correct / total

    return epoch_loss, epoch_acc

In [ ]:
def loop(optimizer_class, save_dir):
    '''
    optimizer:手法
    save_dir:保存先
    '''
    set_seed(SEED)

    if not os.path.exists(f"{save_dir}/result.json"):
        train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True, num_workers = 0)
        test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 0)
        model = CNN_Model().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optimizer_class(model.parameters(), lr = LEARNING_RATE)

        train_loss_history = []
        test_loss_history = []
        train_acc_history = []
        test_acc_history = []
        norm_history = []
        x = []
        fg_count = 0

        for epochs in range(EPOCHS):
            print(f"epoch: {epochs + 1}")

            if epochs == 0:
                train_loss, train_acc = epoch(train_loader, model, criterion)

            else:

                if isinstance(optimizer, SVRG):
                    optimizer.calc_full_grads(model, train_loader, criterion)
                    fg_count += 1

                if isinstance(optimizer, NFG_SVRG) or isinstance(optimizer, ASAI_SVRG):
                    optimizer.init_epoch()
                
                train_loss, train_acc = epoch(train_loader, model, criterion, optimizer)
                fg_count += 1

                if isinstance(optimizer, SVRG) or isinstance(optimizer, NFG_SVRG) or isinstance(optimizer, ASAI_SVRG):
                    fg_count += 1

                if isinstance(optimizer, NFG_SVRG) or isinstance(optimizer, ASAI_SVRG):
                    diff_norm = optimizer.calc_diff_norm(model, train_loader, criterion)
                    norm_history.append(diff_norm)
                    optimizer.end_epoch()

            test_loss, test_acc = epoch(test_loader, model, criterion)

            x.append(fg_count)
            train_loss_history.append(train_loss)
            train_acc_history.append(train_acc)

            test_loss_history.append(test_loss)
            test_acc_history.append(test_acc)

            if (isinstance(optimizer, NFG_SVRG) or isinstance(optimizer, ASAI_SVRG)) and epochs != 0:
                print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} |  Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f} | norm: {diff_norm}")

            else:
                print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} |  Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")

            if fg_count >= EPOCHS:
                break

        if KEEP:
            os.makedirs(save_dir, exist_ok=True)

            tmp = {
                "train_loss": train_loss_history,
                "train_acc": train_acc_history,
                "test_loss": test_loss_history,
                "test_acc": test_acc_history,
                "x": x,
            }
            if isinstance(optimizer, NFG_SVRG) or isinstance(optimizer, ASAI_SVRG):
                tmp["norm"] = norm_history
                
            with open(f"{save_dir}/result.json", "w") as f:
                json.dump(tmp, f, indent = 4)

In [ ]:
loop(optimizer_class = torch.optim.SGD, save_dir = f"{TARGET_DIR}/SGD")

In [ ]:
loop(optimizer_class = SVRG, save_dir = f"{TARGET_DIR}/SVRG")

In [ ]:
loop(optimizer_class = NFG_SVRG, save_dir = f"{TARGET_DIR}/NFG_SVRG")

In [ ]:
loop(optimizer_class = ASAI_SVRG, save_dir = f"{TARGET_DIR}/ASAI_SVRG")

# 結果

In [ ]:
fig = plt.figure(figsize = (12, 12))
fig.suptitle(f"CIFAR10  Learning Rate={LEARNING_RATE}  Batch Size={BATCH_SIZE}" ,fontsize=16)
ax = fig.add_subplot(3, 2, 1)
ax.plot(SGD_x, SGD_train_loss_history, label = "SGD")
ax.plot(SVRG_x, SVRG_train_loss_history, label = "SVRG")
ax.plot(NFG_SVRG_x, NFG_SVRG_train_loss_history, label = "No Full Grad SVRG")
ax.plot(ASAI_SVRG_x, ASAI_SVRG_train_loss_history, label = "ASAI-SVRG", linewidth = 3)
ax.set_xlabel("# full gradient computations")
ax.set_ylabel("loss")
ax.set_title("train loss")
ax.grid()
ax.legend()

ax = fig.add_subplot(3, 2, 2)
ax.plot(SGD_x, SGD_train_acc_history, label = "SGD")
ax.plot(SVRG_x, SVRG_train_acc_history, label = "SVRG")
ax.plot(NFG_SVRG_x, NFG_SVRG_train_acc_history, label = "No Full Grad SVRG")
ax.plot(ASAI_SVRG_x, ASAI_SVRG_train_acc_history, label = "ASAI-SVRG", linewidth = 3)
ax.set_xlabel("# full gradient computations")
ax.set_ylabel("accuracy")
ax.set_title("train accuracy")
ax.grid()
ax.legend()

ax = fig.add_subplot(3, 2, 3)
ax.plot(SGD_x, SGD_test_loss_history, label = "SGD")
ax.plot(SVRG_x, SVRG_test_loss_history, label = "SVRG")
ax.plot(NFG_SVRG_x, NFG_SVRG_test_loss_history, label = "No Full Grad SVRG")
ax.plot(ASAI_SVRG_x, ASAI_SVRG_test_loss_history, label = "ASAI-SVRG", linewidth = 3)
ax.set_xlabel("# full gradient computations")
ax.set_ylabel("loss")
ax.set_title("test loss")
ax.grid()
ax.legend()

ax = fig.add_subplot(3, 2, 4)
ax.plot(SGD_x, SGD_test_acc_history, label = "SGD")
ax.plot(SVRG_x, SVRG_test_acc_history, label = "SVRG")
ax.plot(NFG_SVRG_x, NFG_SVRG_test_acc_history, label = "No Full Grad SVRG")
ax.plot(ASAI_SVRG_x, ASAI_SVRG_test_acc_history, label = "ASAI-SVRG", linewidth = 3)
ax.set_xlabel("# full gradient computations")
ax.set_ylabel("accuracy")
ax.set_title("test accuracy")
ax.grid()
ax.legend()

ax = fig.add_subplot(3, 2, 5)
ax.plot(NFG_SVRG_x , NFG_SVRG_norm_history, label = "No Full Grad SVRG", c='green', linewidth = 2)
ax.plot(ASAI_SVRG_x, ASAI_SVRG_norm_history, label = "ASAI-SVRG", c='red', linewidth = 2)
ax.set_xlabel("#grad / n")
ax.set_ylabel("norm")
ax.set_title("full grad diff norm")
ax.grid()
ax.legend()

plt.tight_layout()
plt.show()